# Why do Alzheimer's trials stop?

In [141]:
import pandas as pd
import sys
sys.path.append("..")
df = pd.read_csv("../data/processed/alzheimer.csv")

I have taken 3 types of trials because trials being stopped does not mean only terminated but withdrawn and suspended as well

In [142]:
stop = pd.Series(df["status"])
mask = stop.isin(["TERMINATED","SUSPENDED","WITHDRAWN"])
stopped = df[mask].copy()
stopped

,nct_id,title,status,phases,study_type,startdate,why_stopped,enrollment,completion_date,sponsor_type,has_results
12,NCT05310071,A Study to Verify the Clinical Benefit of Aduc...,TERMINATED,PHASE3,INTERVENTIONAL,2022-06-02,Sponsor's Decision,1027.0,2024-08-12,INDUSTRY,True
15,NCT03980730,Study of Azeliragon in Patients With Mild Alzh...,TERMINATED,PHASE2,INTERVENTIONAL,2019-06-27,for business reasons,43.0,2021-01-29,INDUSTRY,True
18,NCT01979432,The Cognitive and Cardiovascular Health in the...,TERMINATED,NaN,INTERVENTIONAL,2013-12,Lack of patients,17.0,2015-12-12,OTHER,False
35,NCT01560585,Open Label Study of Isotretinoin in Mild to Mo...,TERMINATED,"PHASE1, PHASE2",INTERVENTIONAL,2012-04,Adverse events in 2/3 participants; study was ...,3.0,2014-08,OTHER,True
37,NCT02263235,In Vivo Alzheimer Proteomics,TERMINATED,NaN,INTERVENTIONAL,2013-10-08,end of the inclusion period,89.0,2018-05-22,OTHER,False
...,...,...,...,...,...,...,...,...,...,...,...
4052,NCT02912169,Study to Assess the Safety and Effects of Auto...,WITHDRAWN,"PHASE1, PHASE2",INTERVENTIONAL,2015-11-01,company dissolved,0.0,2017-11-15,INDUSTRY,False
4059,NCT00287742,A Study of the Effectiveness and Safety of Ris...,TERMINATED,PHASE3,INTERVENTIONAL,2002-03,A decision was made to discontinue the study d...,33.0,2003-03,INDUSTRY,False
4063,NCT00912288,A Phase 3 Efficacy Study Of Dimebon In Patient...,TERMINATED,PHASE3,INTERVENTIONAL,2009-09,See termination reason in detailed description.,86.0,2010-08,INDUSTRY,True
4085,NCT01568086,Follow-up Study to Evaluate Long-term Safety a...,TERMINATED,NaN,OBSERVATIONAL,2011-12,The study could not be performed as planned fo...,16.0,2013-10,INDUSTRY,False


In [143]:
stopped.shape

(413, 11)

The amount of stopped trials has changed from 304 to 413 as we have increased the amount of stopped trials with the addition of suspended and withdrawn

In [144]:
stopped["status"].value_counts()
stopped.head(20)

,nct_id,title,status,phases,study_type,startdate,why_stopped,enrollment,completion_date,sponsor_type,has_results
12,NCT05310071,A Study to Verify the Clinical Benefit of Aduc...,TERMINATED,PHASE3,INTERVENTIONAL,2022-06-02,Sponsor's Decision,1027.0,2024-08-12,INDUSTRY,True
15,NCT03980730,Study of Azeliragon in Patients With Mild Alzh...,TERMINATED,PHASE2,INTERVENTIONAL,2019-06-27,for business reasons,43.0,2021-01-29,INDUSTRY,True
18,NCT01979432,The Cognitive and Cardiovascular Health in the...,TERMINATED,NaN,INTERVENTIONAL,2013-12,Lack of patients,17.0,2015-12-12,OTHER,False
35,NCT01560585,Open Label Study of Isotretinoin in Mild to Mo...,TERMINATED,"PHASE1, PHASE2",INTERVENTIONAL,2012-04,Adverse events in 2/3 participants; study was ...,3.0,2014-08,OTHER,True
37,NCT02263235,In Vivo Alzheimer Proteomics,TERMINATED,NaN,INTERVENTIONAL,2013-10-08,end of the inclusion period,89.0,2018-05-22,OTHER,False
40,NCT00916617,Study Evaluating The Long-Term Safety And Effi...,TERMINATED,PHASE2,INTERVENTIONAL,2009-06,"The study was terminated on August 6, 2012, be...",62.0,2012-09,INDUSTRY,True
47,NCT05402423,An Integrated Intervention of Computerized Cog...,WITHDRAWN,NaN,INTERVENTIONAL,2022-06-27,It was not possible to acquire the medical device,0.0,2023-06-26,OTHER,False
48,NCT03484143,Neuro RX Gamma - Pivotal Phase,SUSPENDED,NaN,INTERVENTIONAL,2019-06-26,Recruitment has been too slow. We are consider...,228.0,2023-05,INDUSTRY,False
54,NCT02791191,A Study of LY3202626 on Disease Progression in...,TERMINATED,PHASE2,INTERVENTIONAL,2016-06-16,Low likelihood of identifying a statistically ...,316.0,2018-07-02,INDUSTRY,True
56,NCT02781220,Implications for Management of PET Amyloid Cla...,TERMINATED,NaN,OBSERVATIONAL,2016-07,IRB Study Closure - No HRPP Progress Update,69.0,2023-06-10,OTHER,False


Going through 20 studies, looking at why they were stopped(why_stopped) made me realize that almost every reason is written differently even when its the same idea. So I cannot count them as they are.

Cleaning the data

In [145]:
stopped["cleaned_data"] =stopped["why_stopped"].str.lower().str.strip()

In [146]:
(stopped["cleaned_data"].str.contains("fund",na=False)).sum()


np.int64(31)

funding ->sponsor,business,fund
recruitment -> enrollment ,recruit,accrual,participants
efficacy  -> efficacy,futility
safety -> safety,adverse

In [200]:
def categorize_text(text):
    if pd.isna(text):
        return "missing"
    if "pi " in text or "investigator" in text or "left" in text:
        return "investigator"
    if "covid" in text:
        return "covid"
    if "administrative" in text:
        return "administrative"
    if "enrol" in text or "recruit" in text or"accrual" in text or "particip" in text or "patient" in text:
        return "recruitment"
    if "efficacy" in text or "futility" in text:
        return "efficacy"
    if "safety" in text or "adverse" in text:
        return "safety"
    if "sponsor" in text or "business" in text or "fund" in text or "strateg" in text:
        return "funding"   
    if "time" in text or "period" in text:
        return "time_managment"
    if "description" in text:
        return "see description"
    else:
        return "other"
    


In [201]:
stopped["reason_category"] = stopped["cleaned_data"].apply(categorize_text)

,nct_id,title,status,phases,study_type,startdate,why_stopped,enrollment,completion_date,sponsor_type,has_results,cleaned_data,reason_category
12,NCT05310071,A Study to Verify the Clinical Benefit of Aduc...,TERMINATED,PHASE3,INTERVENTIONAL,2022-06-02,Sponsor's Decision,1027.0,2024-08-12,INDUSTRY,True,sponsor's decision,funding
15,NCT03980730,Study of Azeliragon in Patients With Mild Alzh...,TERMINATED,PHASE2,INTERVENTIONAL,2019-06-27,for business reasons,43.0,2021-01-29,INDUSTRY,True,for business reasons,funding
18,NCT01979432,The Cognitive and Cardiovascular Health in the...,TERMINATED,NaN,INTERVENTIONAL,2013-12,Lack of patients,17.0,2015-12-12,OTHER,False,lack of patients,other
35,NCT01560585,Open Label Study of Isotretinoin in Mild to Mo...,TERMINATED,"PHASE1, PHASE2",INTERVENTIONAL,2012-04,Adverse events in 2/3 participants; study was ...,3.0,2014-08,OTHER,True,adverse events in 2/3 participants; study was ...,recruitment
37,NCT02263235,In Vivo Alzheimer Proteomics,TERMINATED,NaN,INTERVENTIONAL,2013-10-08,end of the inclusion period,89.0,2018-05-22,OTHER,False,end of the inclusion period,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4052,NCT02912169,Study to Assess the Safety and Effects of Auto...,WITHDRAWN,"PHASE1, PHASE2",INTERVENTIONAL,2015-11-01,company dissolved,0.0,2017-11-15,INDUSTRY,False,company dissolved,other
4059,NCT00287742,A Study of the Effectiveness and Safety of Ris...,TERMINATED,PHASE3,INTERVENTIONAL,2002-03,A decision was made to discontinue the study d...,33.0,2003-03,INDUSTRY,False,a decision was made to discontinue the study d...,other
4063,NCT00912288,A Phase 3 Efficacy Study Of Dimebon In Patient...,TERMINATED,PHASE3,INTERVENTIONAL,2009-09,See termination reason in detailed description.,86.0,2010-08,INDUSTRY,True,see termination reason in detailed description.,other
4085,NCT01568086,Follow-up Study to Evaluate Long-term Safety a...,TERMINATED,NaN,OBSERVATIONAL,2011-12,The study could not be performed as planned fo...,16.0,2013-10,INDUSTRY,False,the study could not be performed as planned fo...,other


In [202]:
(stopped["reason_category"].value_counts())

reason_category
other              110
recruitment         89
missing             58
funding             44
efficacy            37
covid               26
investigator        23
safety               9
administrative       7
see description      6
time_managment       4
Name: count, dtype: int64

In [180]:
mask = stopped["reason_category"] == "other"
other_category = stopped[mask]
other_category

,nct_id,title,status,phases,study_type,startdate,why_stopped,enrollment,completion_date,sponsor_type,has_results,cleaned_data,reason_category
37,NCT02263235,In Vivo Alzheimer Proteomics,TERMINATED,NaN,INTERVENTIONAL,2013-10-08,end of the inclusion period,89.0,2018-05-22,OTHER,False,end of the inclusion period,other
47,NCT05402423,An Integrated Intervention of Computerized Cog...,WITHDRAWN,NaN,INTERVENTIONAL,2022-06-27,It was not possible to acquire the medical device,0.0,2023-06-26,OTHER,False,it was not possible to acquire the medical device,other
54,NCT02791191,A Study of LY3202626 on Disease Progression in...,TERMINATED,PHASE2,INTERVENTIONAL,2016-06-16,Low likelihood of identifying a statistically ...,316.0,2018-07-02,INDUSTRY,True,low likelihood of identifying a statistically ...,other
56,NCT02781220,Implications for Management of PET Amyloid Cla...,TERMINATED,NaN,OBSERVATIONAL,2016-07,IRB Study Closure - No HRPP Progress Update,69.0,2023-06-10,OTHER,False,irb study closure - no hrpp progress update,other
64,NCT06217146,A Medical Cannabis Oil for Treatment of Agitat...,TERMINATED,NaN,INTERVENTIONAL,2022-10-12,Part I: Open-label was completed.,24.0,2024-03-10,INDUSTRY,False,part i: open-label was completed.,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4000,NCT02221947,"Study to Evaluate the Preliminary Safety, Effi...",TERMINATED,"PHASE1, PHASE2",INTERVENTIONAL,2014-06,"Part 2 of study replaced by NTRP-101-202, asse...",9.0,2014-12,INDUSTRY,True,"part 2 of study replaced by ntrp-101-202, asse...",other
4038,NCT04121208,MIcroglial Colony Stimulating Factor-1 Recepto...,TERMINATED,PHASE1,INTERVENTIONAL,2021-04-07,Delayed due to COVID19. Study drug expired bef...,2.0,2022-02-18,OTHER,False,delayed due to covid19. study drug expired bef...,other
4052,NCT02912169,Study to Assess the Safety and Effects of Auto...,WITHDRAWN,"PHASE1, PHASE2",INTERVENTIONAL,2015-11-01,company dissolved,0.0,2017-11-15,INDUSTRY,False,company dissolved,other
4085,NCT01568086,Follow-up Study to Evaluate Long-term Safety a...,TERMINATED,NaN,OBSERVATIONAL,2011-12,The study could not be performed as planned fo...,16.0,2013-10,INDUSTRY,False,the study could not be performed as planned fo...,other


In [193]:

other_category["cleaned_data"].str.split().explode().value_counts().head(50)

cleaned_data
the               86
study             64
to                61
of                60
was               32
in                25
not               23
due               22
and               18
a                 17
terminated        14
no                13
this              13
has               13
is                12
pi                11
with              11
be                 9
on                 9
data               9
trial              8
for                8
been               8
administrative     8
clinical           8
did                8
an                 7
meet               7
primary            7
subjects           7
new                6
company            6
by                 6
research           6
could              6
never              6
pandemic           6
results            6
study.             6
as                 6
there              5
suspended          5
between            5
discontinued       5
that               5
its                5
covid              5


In [ ]:
(other_category["cleaned_data"].str.contains("administrative",na=False)).sum()


np.int64(8)

In [199]:
mask = other_category["cleaned_data"].str.contains("administrative")
administrative =  other_category[mask]
administrative

,nct_id,title,status,phases,study_type,startdate,why_stopped,enrollment,completion_date,sponsor_type,has_results,cleaned_data,reason_category
618,NCT04031469,A Non-Interventional Pilot Study to Explore th...,SUSPENDED,NaN,OBSERVATIONAL,2019-07-11,Administrative,1000.0,2030-12-10,OTHER,False,administrative,other
713,NCT01815112,Early Diagnosis of Alzheimer-like Dementia: Be...,TERMINATED,NaN,INTERVENTIONAL,2008-02,Administrative re-organization of the departme...,60.0,2014-04,OTHER,False,administrative re-organization of the departme...,other
972,NCT06135051,Ultrasonic Neuromodulation for Treatment of Co...,TERMINATED,"PHASE1, PHASE2",INTERVENTIONAL,2024-08-18,IRB administrative closure,1.0,2024-08-21,OTHER,False,irb administrative closure,other
1539,NCT02166827,"Effects of a NeuroAD System, for the Treatment...",TERMINATED,NaN,INTERVENTIONAL,2014-11,Administrative decision,1.0,2015-11,INDUSTRY,False,administrative decision,other
1560,NCT01626391,Safety Study of TRx0237 in Patients Already Ta...,TERMINATED,PHASE2,INTERVENTIONAL,2012-09,This study has been terminated for administrat...,9.0,2013-03,INDUSTRY,True,this study has been terminated for administrat...,other
2747,NCT01669876,Study to Evaluate Dietary Supplementation With...,TERMINATED,PHASE2,INTERVENTIONAL,2012-08,Administrative status of active ingredient ana...,85.0,2015-02,INDUSTRY,False,administrative status of active ingredient ana...,other
3751,NCT01825577,Exploring the Use of Transdermal Methylphenida...,TERMINATED,PHASE4,INTERVENTIONAL,2012-11,Administrative Reasons,14.0,2015-02,OTHER,True,administrative reasons,other
3889,NCT02245568,Open-Label Study of Leuco-methylthioninium Bis...,TERMINATED,PHASE3,INTERVENTIONAL,2014-08,This study has been terminated for administrat...,913.0,2017-05,INDUSTRY,True,this study has been terminated for administrat...,other
